# DIT5411 — Traditional Chinese Handwriting Recognition
**Google Drive version** — dataset, checkpoints and results are stored on Drive so they survive runtime resets. Enable GPU: Runtime → Change runtime type → T4 GPU.

In [ ]:
import os, sys, subprocess

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isfile('train.py'):
        if not os.path.isdir('DIT5411-LuiYuHin'):
            subprocess.run(['git', 'clone', '-q', 'https://github.com/cveg1234/DIT5411-LuiYuHin.git'], check=True)
        os.chdir('DIT5411-LuiYuHin')
    subprocess.run(['git', 'pull', '-q'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
    DRIVE_ROOT = '/content/drive/MyDrive/DIT5411'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    DATA_DIR = DRIVE_ROOT + '/data/cleaned_data'
    OUT_DIR = DRIVE_ROOT + '/results'
else:
    if not os.path.isfile('train.py'):
        os.chdir('..')
    DATA_DIR = 'data/cleaned_data'
    OUT_DIR = 'results'

os.makedirs(OUT_DIR, exist_ok=True)
print('Working dir :', os.getcwd())
print('Dataset    :', DATA_DIR)
print('Checkpoints:', OUT_DIR + '/checkpoints')

## Dataset (one time)
Stored on Google Drive at `MyDrive/DIT5411/data/cleaned_data`, so it survives runtime resets.

In [ ]:
if not os.path.isdir(DATA_DIR) or len(os.listdir(DATA_DIR)) < 1000:
    if not os.path.isdir('handwritting_data_all'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/chenkenanalytic/handwritting_data_all.git'], check=True)
    import glob
    with open('all_data.zip', 'wb') as out:
        for part in sorted(glob.glob('handwritting_data_all/all_data.zip.*')):
            with open(part, 'rb') as f:
                out.write(f.read())
    subprocess.run([sys.executable, 'extract_dataset.py', '--zip', 'all_data.zip', '--dest', DATA_DIR], check=True)
print('Dataset folders:', len(os.listdir(DATA_DIR)))

## Quick pilot (optional, 2 epochs)

In [ ]:
subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                '--output-dir', OUT_DIR + '/pilot',
                '--num-classes', '20', '--epochs', '2', '--models', 'A', 'B',
                '--no-imagenet', '--steps-per-epoch', '100'])

## Full assignment run
Checkpoints + results go to Google Drive. If the runtime disconnects or GPU quota runs out, run the **setup cell** (remounts Drive) then this cell — `--resume` continues from the last saved epoch.

In [ ]:
subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                '--output-dir', OUT_DIR,
                '--epochs', '5',
                '--models', 'A', 'B', 'C', '--resume'])

In [ ]:
from IPython.display import Markdown, display
p = OUT_DIR + '/RESULTS.md'
display(Markdown(open(p, encoding='utf-8').read()) if os.path.isfile(p) else 'RESULTS.md not generated yet — finish training first.')